# 04. 대화형 Agent 완성

---
## 0. 설치 & 경로

In [1]:
!pip install -U duckduckgo-search

In [28]:
!pip install openai python-dotenv langchain langchain-openai langchain-core langchain-community langchain-text-splitters langchain-classic faiss-cpu pypdf pytz yfinance requests ddgs tabulate rank_bm25 langchain-chroma

In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

RAG_SAMPLES = os.path.join(os.getcwd(), 'samples', 'pdf_samples')
print('RAG_SAMPLES:', RAG_SAMPLES)
for p in sorted(Path(RAG_SAMPLES).glob('*.pdf')):
    print(' -', p.name)

RAG_SAMPLES: c:\Users\Admin\Desktop\실습\14일차\samples\pdf_samples
 - A survey on large language model based autonomous agents.pdf
 - data2vec.pdf
 - NeurIPS-2023-analyzing-vision-transformers-for-image-classification-in-class-embedding-space-Paper-Conference.pdf
 - Physics-Guided Deep Learning.pdf
 - supervised-contrastive-learning-Paper.pdf
 - 보도자료.pdf
 - 학칙.pdf


---
## Step 1. RAG VectorStore 

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import os

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 학칙.pdf
loader = PyPDFLoader(os.path.join(RAG_SAMPLES, '학칙.pdf'))
all_splits = text_splitter.split_documents(loader.load())

# news.pdf
loader_news = PyPDFLoader(os.path.join(RAG_SAMPLES, '보도자료.pdf'))
news_splits = text_splitter.split_documents(loader_news.load())
all_splits.extend(news_splits)

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large', api_key=OPENAI_API_KEY
)

persist_directory = 'chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

Loading existing Chroma store


In [5]:
retriever # 생성한 VectorStore 기반으로 검색을 수행하는 Retriever 객체

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001FC5A263470>, search_kwargs={'k': 3})

---
## Step 1.1 검색 품질 점검

`석사학위과정 수업연한`처럼 **별표(부록)에 같은 단어가 반복**되는 PDF에서는, 기본 유사도 검색이 엉뚱한 청크 3개만 가져올 수 있습니다.  
아래에서 검색 결과와 **실제 정답 조항**을 비교해 봅니다.

In [6]:
demo_query = '석사학위과정 수업연한'

retriever_baseline = vectorstore.as_retriever(search_kwargs={'k': 3})
baseline_hits = retriever_baseline.invoke(demo_query)

print(f'질문: {demo_query}\n')
for i, doc in enumerate(baseline_hits, 1):
    has_answer = '수업연한' in doc.page_content and '2년(4학기)' in doc.page_content
    print(f'[{i}] 수업연한 정답 포함: {has_answer}')
    print(doc.page_content[:350].replace('\n', ' '))
    print('-' * 60)

질문: 석사학위과정 수업연한

[1] 수업연한 정답 포함: False
34 [별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드 문과대학 국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German Language and LiteraturePANTONE 5315C불어불문학과문학석사M.A. in French Language
------------------------------------------------------------
[2] 수업연한 정답 포함: False
협동과정 비교문학협동과정문학석사M.A. in Comparative LiteraturePANTONE 5315C인지과학협동과정인지과학석사M.S. in Cognitive SciencePANTONE 5315C문학석사M.A. in Cognitive SciencePANTONE 5315C공학석사M.S. in Cognitive Science and EngineeringPANTONE 172C경영학석사M.A. in Cognitive Science and ManagementPANTONE 477C이학석사M.S. in Cognitive SciencePANTONE 136C한국학협동과정문학석사M.A. in Korean StudiesPANTONE 
------------------------------------------------------------
[3] 수업연한 정답 포함: False
Ⅰ. 대학원 학칙  /  29 [별표 1]학위과정별(캠퍼스) 입학정원 현황(2026학년도)□ 일반 학위과정캠퍼스석사 정원(명)박사 정원(명)신촌1,3281)1,0292

In [7]:
# 학칙.pdf 본문에 실제로 있는 조항 (검색이 찾아야 할 정답)
answer_chunks = [
    s for s in all_splits
    if '수업연한' in s.page_content and '2년(4학기)' in s.page_content
]

print(f'정답 청크 수: {len(answer_chunks)}')
print('=' * 60)
print(answer_chunks[0].page_content[:700])

정답 청크 수: 1
③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학연한에 산입하지 아니한다. 1. 석사학위과정: 4년(8학기)2. 박사학위과정: 7년(14학기)3. 통합과정: 8년(16학기)② 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자로서 절차에 따라 학적에 구분표시 되어 있는 학생에 대하여는 재학연한을 제한하지 아니한다.제2조의9(융합전공) ① 대학원 내 두 개 이상의 학과가 융합적 교과과정(융합전공)을 공동 운영할 수 있다.② 융합전공의 운영에 관한 사항은 별도로 정한다.제2조의10(학생설계전공) ① 새로운 학문수요의 충족과 자기 주도적 학습능력의 향상을 위하여 학생설계전공을 둘 수 있다.② 학생설계전공의 설치 및 이수요건은 별도로 정한다.제3조(입학정원) ① 대학원에 두는 학위과정별 입학정원([별표 1])은 대학원운영위원회에서 대학별로 정한다. 단, 통합과정 입학정원은 박사학위과정


---
## Step 1.2 BM25 키워드 검색 + 하이브리드

**BM25**는 단어(키워드) 빈도 기반 검색입니다. `수업연한`처럼 **문서에 정확히 등장하는 용어**를 찾을 때 벡터 검색보다 유리한 경우가 많습니다.

| 방식 | 장점 | 약점 |
|------|------|------|
| 벡터 검색 | 의미가 비슷한 문장도 찾음 | 별표처럼 같은 단어가 반복되면 엉뚱한 청크가 올라옴 |
| BM25 | 키워드가 들어 있는 청크를 직접 찾음 | 한글 PDF는 띄어쓰기가 적어 **토크나이저 설정**이 필요 |

실습에서는 **BM25 + 벡터**를 `EnsembleRetriever`로 합쳐 하이브리드 검색을 씁니다.

In [8]:
import re
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.tools import create_retriever_tool


def ko_preprocess(text):
    """한글 학칙 PDF용 — 띄어쓰기 대신 한글 어절 단위로 토큰 분리"""
    return re.findall(r'[가-힣]+|\d+', text)


def show_hits(label, hits, preview=3):
    print(f'\n=== {label} ===')
    for i, doc in enumerate(hits[:preview], 1):
        has_answer = '수업연한' in doc.page_content and '2년(4학기)' in doc.page_content
        print(f'[{i}] 수업연한 정답 포함: {has_answer}')
        print(doc.page_content[:280].replace('\n', ' '))
        print('-' * 50)

In [9]:
# BM25 키워드 검색
bm25_retriever = BM25Retriever.from_documents(all_splits, preprocess_func=ko_preprocess)
bm25_retriever.k = 3
bm25_hits = bm25_retriever.invoke(demo_query)
show_hits('BM25 키워드 검색 (k=3)', bm25_hits)


=== BM25 키워드 검색 (k=3) ===
[1] 수업연한 정답 포함: True
③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학
--------------------------------------------------
[2] 수업연한 정답 포함: False
34 [별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드 문과대학 국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German
--------------------------------------------------
[3] 수업연한 정답 포함: False
10 제7조의2(편입학) ① 당해 학년도 입학정원에서 재학생 수를 뺀 범위 안에서 편입학생을 모집할 수 있다.② 편입학에 관한 내규는 따로 정한다.제7조의3(재입학) 제적 또는 과정수료 처리된 자는 입학정원의 결원이 있을 때 1회에 한하여 재입학을 신청할 수 있으며, 이에 관한 사항은 따로 정한다.1. <삭제>2. <삭제>3. <삭제>제4장 등록 및 학적 변동제8조(등록) ① 대학원 각 학위과정의 학생은 재학연한 기간 내에 다음 각 

In [10]:
# 하이브리드: BM25 + 벡터
vector_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],  #키워드 등장에 중점을 두려면 BM25쪽에 가중치를 늘리면 된다.
)
hybrid_hits = hybrid_retriever.invoke(demo_query)
show_hits('하이브리드 BM25 + 벡터 (k=3)', hybrid_hits)

found = any(
    '수업연한' in d.page_content and '2년(4학기)' in d.page_content
    for d in hybrid_hits
)
print('\n수업연한 정답 청크 포함 여부:', found)


=== 하이브리드 BM25 + 벡터 (k=3) ===
[1] 수업연한 정답 포함: False
34 [별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드 문과대학 국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German
--------------------------------------------------
[2] 수업연한 정답 포함: True
③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학
--------------------------------------------------
[3] 수업연한 정답 포함: False
협동과정 비교문학협동과정문학석사M.A. in Comparative LiteraturePANTONE 5315C인지과학협동과정인지과학석사M.S. in Cognitive SciencePANTONE 5315C문학석사M.A. in Cognitive SciencePANTONE 5315C공학석사M.S. in Cognitive Science and EngineeringPANTONE 172C경영학석사M.A. in Cognitive Scienc

---
## Step 2. Retriever → Tool (5.3 마지막 섹션)

`langchain_core.tools.create_retriever_tool` — 5.3과 동일합니다.

> **Step 1.2**에서 하이브리드 retriever로 `search_pdf_tool`을 이미 만들었다면, 이 셀은 생략해도 됩니다.

In [11]:
from langchain_core.tools import create_retriever_tool

search_pdf_tool = create_retriever_tool(
    retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)

#이건 @Tool을 안붙여도 LLM이 tool로 인식한다. langchain_core.tools의 기능임.

---
## Step 3. 유틸 Tool (5.2 복습)

시각 · 주가 — `@tool` 로 등록합니다.

In [12]:
from datetime import datetime
import pytz
import yfinance as yf
from langchain_core.tools import tool

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각. timezone 예: Asia/Seoul, location 예: 서울"""
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime('%Y-%m-%d %H:%M:%S')
    return f'{timezone} ({location}) 현재시각 {now}'

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """주식 가격 조회. ticker 예: TSLA, period 예: 1mo"""
    history = yf.Ticker(ticker).history(period=period)
    return history.tail(3).to_string() if not history.empty else '데이터 없음'


In [13]:
print(get_current_time.invoke({'timezone': 'Asia/Seoul', 'location': '서울'}))

Asia/Seoul (서울) 현재시각 2026-07-09 14:48:56


---
## Step 4. DuckDuckGo 웹 검색

최신 뉴스·시사는 PDF RAG가 아니라 **웹 검색** Tool을 씁니다.  
Agent가 질문마다 `search_pdf_documents` vs `duckduckgo_results_json` 중 선택합니다.

In [14]:
from langchain_community.tools import DuckDuckGoSearchResults

web_search = DuckDuckGoSearchResults(num_results=3)
print(web_search.name, '-', web_search.description[:80], '...')

duckduckgo_results_json - A wrapper around Duck Duck Go Search. Useful for when you need to answer questio ...


In [15]:
web_search.invoke("26년 월드컵 한국 vs 멕시코 점수는?")

#검색한 내용을 LLM이 내용을 요약하여 대답한다.

'snippet: 분류 홍명보호 (성인 2기)/2026 FIFA 월드컵 멕시코 축구 국가대표팀 경기 한국-멕시코 관계 2026년 축구 경기, title: 홍명보호(성인 2기)/2026 Fifa 월드컵/멕시코전 - 나무위키, link: https://namu.wiki/w/홍명보호(성인+2기)/2026+FIFA+월드컵/멕시코전, snippet: 이번엔 2026 월드컵 2차전 한국 vs 멕시코에 대한 모든 정보와 관전 포인트를 정리 하겠습니다. 체코를 2-1로 뒤집은 역전승의 여운이 채 가시기도 전에 이제 진짜 고비가 다가왔습니다. 6월 19일 오전 10시, 같은 장소인 멕시코 과달라하라에서 이번엔 개최국 멕시코를 상대합니다. 체코전이 "반드시 ..., title: 2026 월드컵 한국 vs 멕시코 완벽 분석 (시간, 피파 랭킹 순위, 역대 전적, 중계) : 네이버 블로그, link: https://blog.naver.com/nyria99/224316761733, snippet: 대한민국 축구대표팀은 2026 북중미 월드컵 A조 2차전에서 공동 개최국 멕시코와 맞붙습니다. 경기 시간은 한국시간 6월 19일 금요일 오전 10시이며 장소는 멕시코 과달라하라 스타디움입니다. 한국과 멕시코는 1차전에서 나란히 승리해 승점 3점을 확보했기 때문에 이번 맞대결 결과가 조 선두와 32강 ..., title: 대한민국 vs 멕시코 역대 전적, 월드컵 조별리그 경기 일정, link: https://www.infogoodman.com/2026/06/korea-mexico-head-to-head-world-cup-schedule.html'

---
## Step 5. AgentExecutor — Tool 통합

5.2 수동 tool 루프 대신 **AgentExecutor**가 tool_calls 루프를 자동 처리합니다.

In [16]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

agent_tools = [get_current_time, get_yf_stock_history, search_pdf_tool, web_search]
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

['get_current_time', 'get_yf_stock_history', 'search_pdf_documents', 'duckduckgo_results_json']


In [17]:
# PDF → search_pdf_documents
r = executor.invoke({'input': '석사학위과정 수업연한은?', 'chat_history': []})
print('RAG:', r['output'])

#벡터기반 모델이라 제대로 못찾는 모습.



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사학위과정 수업연한'}`


34
[별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드
문과대학
국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German Language and LiteraturePANTONE 5315C불어불문학과문학석사M.A. in French Language and LiteraturePANTONE 5315C노어노문학과문학석사M.A. in Russian Language and LiteraturePANTONE 5315C사학과문학석사M.A. in HistoryPANTONE 5315C철학과문학석사M.A. in PhilosophyPANTONE 5315C문헌정보학과문헌정보학석사M.A. in Library and Information SciencePANTONE 5315C기록관리학석사M.A. in Record Management and ArchivesPANTONE 5315C심리학과문학석사M.A. in PsychologyPANTONE 5315C심리학석사M.S. in PsychologyPANTONE 5315C인지과학석사M.S. in Cognitive SciencePANTONE 5315C
협동과정

협동과정
비교문학협동과정문학석사M.A. in Comparative LiteraturePANTONE 5315C인지과학협동과정인지과학석사M.S. in Cognitive SciencePANTONE 5315C문학석사M.A. in Cognit

In [18]:
search_pdf_tool = create_retriever_tool(
    hybrid_retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)
# PDF → search_pdf_documents

agent_tools = [get_current_time, get_yf_stock_history, search_pdf_tool, web_search]
agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

r = executor.invoke({'input': '석사학위과정 수업연한은?', 'chat_history': []})
print('RAG:', r['output'])



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사학위과정 수업연한'}`


34
[별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드
문과대학
국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German Language and LiteraturePANTONE 5315C불어불문학과문학석사M.A. in French Language and LiteraturePANTONE 5315C노어노문학과문학석사M.A. in Russian Language and LiteraturePANTONE 5315C사학과문학석사M.A. in HistoryPANTONE 5315C철학과문학석사M.A. in PhilosophyPANTONE 5315C문헌정보학과문헌정보학석사M.A. in Library and Information SciencePANTONE 5315C기록관리학석사M.A. in Record Management and ArchivesPANTONE 5315C심리학과문학석사M.A. in PsychologyPANTONE 5315C심리학석사M.S. in PsychologyPANTONE 5315C인지과학석사M.S. in Cognitive SciencePANTONE 5315C
협동과정

③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 

In [19]:
# 시각 → get_current_time
r = executor.invoke({'input': '청주 지금 몇시야?', 'chat_history': []})
print('시각:', r['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_time` with `{'timezone': 'Asia/Seoul', 'location': '청주'}`


Asia/Seoul (청주) 현재시각 2026-07-09 14:49:45청주 현재 시각은 2026년 7월 9일 14시 49분 45초입니다.

> Finished chain.
시각: 청주 현재 시각은 2026년 7월 9일 14시 49분 45초입니다.


In [20]:
r = executor.invoke({'input': '2026 월드컵 대한민국 경기 결과들 알려줘.', 'chat_history': []})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_results_json` with `{'query': '2026 월드컵 대한민국 경기 결과'}`


snippet: 대한민국은 본선에 처음 진출한 1954 스위스 월드컵 을 포함해 72년 역사상 월드컵 단일 대회 최저 순위 (34위) 로 이번 대회를 마치게 되었다. [1] [2] 아무리 첫 경기에서 1승을 거두었다 한들 순위 기준으로는 명백히 대한민국 축구 역사상 최악의 월드컵 이며, 이번 월드컵에 진출한 국가들 중 튀르키예 ..., title: 2026 Fifa 월드컵/팀별 결산/대한민국 - 나무위키, link: https://namu.wiki/w/2026+FIFA+월드컵/팀별+결산/대한민국, snippet: 2026 북중미 월드컵 조별 순위(A~L조)와 경기 결과를 실시간 확인하세요. 대한민국이 속한 A조 순위, 조 편성, 대진표, 순위 결정 기준과 32강 진출 조건까지 정리했습니다., title: 2026 북중미 월드컵 조별 순위 경기 결과 확인 - Xocowriter, link: https://xocowriter.com/2026-world-cup-group-standings-results/, snippet: 2026 북중미 월드컵, 한국은 지금 어디까지 왔을까요? 1차전 승리, 2차전 패배를 지나 운명의 최종전을 앞둔 대한민국 대표팀의 전체 일정을 결과와 함께 한국시간 기준으로 깔끔하게 정리했습니다. 멕시코는 이미 조 1위로 32강을 확정했습니다. 한국은 남아공전에서 이기거나 비기면 조 2위로 32강 진출 입니다. 이번 대회는 골득실보다 승자승이 우선이라, 1차전에서 체코를 잡은 한국은 비겨도 2위를 지킵니다. 다만 지면 3위 경합으로 밀려 복잡한 경우의 수에 빠집니다. ※ 32강 이후 대진·날짜·시간은 조별리그 최종 순위에 따라 확정됩니다. 진출 시 상대와 정확한 킥오프는 중계 편성으로 확인하세요., title: 대한민국 2026 월드컵 경

---
## Step 6. 멀티턴 — `chat_history` (5.1 + Agent)

In [25]:
print(executor.invoke({'input': '학칙에서 석사 수업연한 알려줘', 'chat_history': []})['output'])
print(executor.invoke({'input': '방금 말한 내용 한 줄로 다시', 'chat_history': []})['output'])



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사 수업연한'}`


Ⅰ. 대학원 학칙  /  29
[별표 1]학위과정별(캠퍼스) 입학정원 현황(2026학년도)□ 일반 학위과정캠퍼스석사 정원(명)박사 정원(명)신촌1,3281)1,0292)국제12271합계1,4501,1001) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 석사 정원 8명 증원2) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 박사 정원 3명 증원□ 계약학과 학위과정 [정원 외]1. 신촌캠퍼스구분석사 정원(명)박사 정원(명)재교육형10028채용조건형11733합계21761

Ⅰ. 대학원 학칙  /  15

10
제7조의2(편입학) ① 당해 학년도 입학정원에서 재학생 수를 뺀 범위 안에서 편입학생을 모집할 수 있다.② 편입학에 관한 내규는 따로 정한다.제7조의3(재입학) 제적 또는 과정수료 처리된 자는 입학정원의 결원이 있을 때 1회에 한하여 재입학을 신청할 수 있으며, 이에 관한 사항은 따로 정한다.1. <삭제>2. <삭제>3. <삭제>제4장 등록 및 학적 변동제8조(등록) ① 대학원 각 학위과정의 학생은 재학연한 기간 내에 다음 각 항에서 정한 등록을 하여야 하며, 등록은 정규등록과 연구등록으로 구분하고, 등록 시 등록금을 납부하여야 한다.② 정규등록은 각 학위과정 수업연한 내 학기의 등록을 말하며, 연구등록은 각 학위과정의 수업연한을 초과한 학기의 등록을 말한다.③ <삭제>④ 각 학위과정의 학생 중 학위수여 자격 요건을 충족한 학생은 정규등록을 한 학기 단축할 수 있다. 단, 통합과정의 학생 및 석사학위 논문 대체실적을 통해 학위를 취득하고자 하는 자는 제외한다.⑤ <삭제> 제9조(재학생의 권한) 재학생은 학점을 이수할 수 있고, 학위논문 제출 자격시험 등에 응시할 수 있으며, 학위논문 연구지도를 받을 수 있다. 제10조(등록금의 납부 및 반환) ① <삭제>② 등록금 납부

In [21]:
from langchain_core.messages import HumanMessage, AIMessage  #massage를 감싸주는 라이브러리

class ChatSession: #이 세션 안에서는 대화 내용을 누적할 수 있게 한다.
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input': question, 'chat_history': self.history})  #이전 내용을 함께 input함
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('학칙에서 석사 수업연한 알려줘'))
print('2:', session.ask('방금 답한 내용 한 줄로 다시'))



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사 수업연한'}`


Ⅰ. 대학원 학칙  /  29
[별표 1]학위과정별(캠퍼스) 입학정원 현황(2026학년도)□ 일반 학위과정캠퍼스석사 정원(명)박사 정원(명)신촌1,3281)1,0292)국제12271합계1,4501,1001) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 석사 정원 8명 증원2) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 박사 정원 3명 증원□ 계약학과 학위과정 [정원 외]1. 신촌캠퍼스구분석사 정원(명)박사 정원(명)재교육형10028채용조건형11733합계21761

Ⅰ. 대학원 학칙  /  15

10
제7조의2(편입학) ① 당해 학년도 입학정원에서 재학생 수를 뺀 범위 안에서 편입학생을 모집할 수 있다.② 편입학에 관한 내규는 따로 정한다.제7조의3(재입학) 제적 또는 과정수료 처리된 자는 입학정원의 결원이 있을 때 1회에 한하여 재입학을 신청할 수 있으며, 이에 관한 사항은 따로 정한다.1. <삭제>2. <삭제>3. <삭제>제4장 등록 및 학적 변동제8조(등록) ① 대학원 각 학위과정의 학생은 재학연한 기간 내에 다음 각 항에서 정한 등록을 하여야 하며, 등록은 정규등록과 연구등록으로 구분하고, 등록 시 등록금을 납부하여야 한다.② 정규등록은 각 학위과정 수업연한 내 학기의 등록을 말하며, 연구등록은 각 학위과정의 수업연한을 초과한 학기의 등록을 말한다.③ <삭제>④ 각 학위과정의 학생 중 학위수여 자격 요건을 충족한 학생은 정규등록을 한 학기 단축할 수 있다. 단, 통합과정의 학생 및 석사학위 논문 대체실적을 통해 학위를 취득하고자 하는 자는 제외한다.⑤ <삭제> 제9조(재학생의 권한) 재학생은 학점을 이수할 수 있고, 학위논문 제출 자격시험 등에 응시할 수 있으며, 학위논문 연구지도를 받을 수 있다. 제10조(등록금의 납부 및 반환) ① <삭제>② 등록금 납부

---
## Step 7. `input()` 대화 루프

In [23]:
print('종료: exit')
while True:
    user_input = input('사용자: ').strip()
    if user_input.lower() in ('exit', 'quit', '종료'):
        break
    if not user_input:
        continue
    print('AI:', session.ask(user_input))

종료: exit


**AI의 대답만 깔끔하게 띄우기 위한 순서 체크리스트**

1. verbose=False로 executor 만들기
2. session = ChatSession(executor) 다시 실행
3. print(session.executor.verbose) → False 확인
그리고 대화 while 루프 실행

이 순서면 Entering new AgentExecutor chain...은 사라지고 사용자: / AI:만 남습니다.

In [31]:
session.executor.verbose = False  #지금 세션 끄기
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=False, max_iterations=8)  #executor 재생성
session = ChatSession(executor)  #세션도 재생성

print("종료하려면 'exit', 'quit', '종료'를 입력해주세요.")
print('=============================================')
while True:
    user_input = input('사용자: ').strip()
    if user_input.lower() in ('exit', 'quit', '종료'):
        break
    if not user_input:
        continue
    print('사용자:', user_input)
    print('AI:', session.ask(user_input))

종료하려면 'exit', 'quit', '종료'를 입력해주세요.
사용자: 안녕하세요. 저는 권혜선입니다.
AI: 안녕하세요, 권혜선님! 어떻게 도와드릴까요?
사용자: 외쳐 내이름!
AI: 권혜선!


---
## Step 8. 나만의 Tool 만들기

`@tool`로 함수를 등록하고 `agent_tools`에 넣으면 Agent가 사용할 수 있습니다.

1. 함수 정의 (`@tool` + docstring)
2. `.invoke()`로 단독 테스트
3. `agent_tools`에 추가 → prompt 보강 → `executor` 재생성
4. Agent 질문으로 Tool 호출 확인

### 8.1 예시 — `celsius_to_fahrenheit`

섭씨 → 화씨 변환 Tool을 만들고 Agent에 연결합니다.

In [32]:
from langchain_core.tools import tool

@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    """섭씨 온도를 화씨로 변환합니다."""
    fahrenheit = celsius * 9 / 5 + 32
    return f'{celsius}°C = {fahrenheit:.1f}°F'

# 1) Tool 단독 실행
print(celsius_to_fahrenheit.name)
print(celsius_to_fahrenheit.invoke({'celsius': 25}))

celsius_to_fahrenheit
25.0°C = 77.0°F


In [33]:
# 2) agent_tools에 추가 후 Agent 재생성
agent_tools = agent_tools + [celsius_to_fahrenheit]

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time
- 섭씨→화씨 온도 변환 → celsius_to_fahrenheit

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

print('등록된 Tool:', [t.name for t in agent_tools])

등록된 Tool: ['get_current_time', 'get_yf_stock_history', 'search_pdf_documents', 'duckduckgo_results_json', 'celsius_to_fahrenheit']


In [34]:
# 3) Agent가 새 Tool을 호출하는지 확인
r = executor.invoke({'input': '섭씨 36.5도는 화씨로 몇 도야?', 'chat_history': []})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `celsius_to_fahrenheit` with `{'celsius': 36.5}`


36.5°C = 97.7°F섭씨 36.5도는 화씨로 97.7도입니다.

> Finished chain.
섭씨 36.5도는 화씨로 97.7도입니다.


### 8.2 실습 — 나만의 Tool

원하는 기능의 Tool을 직접 만들고, Agent가 호출하는지 확인합니다.

- 예: 단위 변환, 문자열 처리, 간단한 계산, 로컬 파일 읽기 등
- docstring에 **언제 쓸지**를 적어야 Agent가 선택합니다.

In [38]:
# TODO 1) @tool 함수 정의
FRIENDS = {
    "한리원": "여자",
    "임수호": "남자",
    "나서준": "남자",
    "김로아": "여자",
    "정지온": "여자",
    "박지훈": "남자",
}

@tool
def my_custom_tool(name: str) -> str:
    """호수 친구의 이름으로 성별을 조회합니다. 예: name='한리원' 또는 name='지훈'"""
    name = name.strip()

    # 1) 완전 일치
    if name in FRIENDS:
        return f"{name}의 성별은 {FRIENDS[name]}입니다."

    # 2) 부분 일치: 입력한 글자가 이름에 포함되거나, 이름이 입력에 포함
    matches = [full for full in FRIENDS if name in full or full in name]

    if len(matches) == 1:
        full = matches[0]
        return f"{full}의 성별은 {FRIENDS[full]}입니다. (검색어: '{name}')"

    if len(matches) > 1:
        # 동명이인/여러 후보 → 목록을 알려주고 다시 묻게
        options = ", ".join(f"{m}({FRIENDS[m]})" for m in matches)
        return f"'{name}'에 해당하는 친구가 여러 명입니다: {options}. 전체 이름으로 다시 물어보세요."

    known = ", ".join(FRIENDS.keys())
    return f"'{name}'은(는) 친구 목록에 없습니다. 등록된 이름: {known}"

In [39]:
# TODO 2) 단독 실행 테스트
print(my_custom_tool.invoke({"name": "지훈"}))

박지훈의 성별은 남자입니다. (검색어: '지훈')


In [40]:
# TODO 3) agent_tools에 추가 + prompt에 한 줄 안내 + executor 재생성
agent_tools = agent_tools + [my_custom_tool]
prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 현재 시각 → get_current_time
- 주식 가격 → get_yf_stock_history
- 섭씨→화씨 온도 변환 → celsius_to_fahrenheit
- 호수 친구 이름·성별 조회 → my_custom_tool

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

In [ ]:
agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

In [42]:
# TODO 4) Agent 질문으로 Tool 호출 확인
session = ChatSession(executor)  #세션 재생성

print("종료하려면 'exit', 'quit', '종료'를 입력해주세요.")
print('=============================================')
while True:
    user_input = input('사용자: ').strip()
    if user_input.lower() in ('exit', 'quit', '종료'):
        break
    if not user_input:
        continue
    print('사용자:', user_input)
    print('AI:', session.ask(user_input))

종료하려면 'exit', 'quit', '종료'를 입력해주세요.
사용자: 혹시 호수 친구들 중에 지훈이 성별 알아?


> Entering new AgentExecutor chain...

Invoking: `my_custom_tool` with `{'name': '지훈'}`


박지훈의 성별은 남자입니다. (검색어: '지훈')호수 친구들 중에 지훈이는 남자입니다.

> Finished chain.
AI: 호수 친구들 중에 지훈이는 남자입니다.
